In [ ]:
from transformers import AutoProcessor, AutoModelForImageTextToText
import torch

# model_path = "HuggingFaceTB/SmolVLM2-2.2B-Instruct"
model_path = "HuggingFaceTB/SmolVLM2-500M-Video-Instruct"
device = 'mps'

processor = AutoProcessor.from_pretrained(model_path)
smol_model = AutoModelForImageTextToText.from_pretrained(
    model_path,
    dtype=torch.bfloat16,
    # _attn_implementation="flash_attention_2" # gpu only
    _attn_implementation="sdpa" # mps only # xformers 
).to(device)

In [ ]:
import numpy as np

### Commenting on 12s video

In [ ]:
VIDEO_PATH = "/Users/emulie/Data/AUDLClips/ufa_championship_game_clip_trimmed.mp4"

# NOTE: doesn't work because of error: "AttributeError: module 'torchvision.io' has no attribute 'read_video'"
# messages = [
#     {
#         "role": "user",
#         "content": [
#             {"type": "video", "path": VIDEO_PATH},
#             {"type": "text", "text": "You are a ultimate frisbee sports analyst. Describe this video in detail"}
#         ]
#     },
# ]

# inputs = processor.apply_chat_template(
#     messages,
#     add_generation_prompt=True,
#     tokenize=True,
#     return_dict=True,
#     return_tensors="pt",
# ).to(model.device, dtype=torch.bfloat16)

# generated_ids = model.generate(**inputs, do_sample=False, max_new_tokens=64)
# generated_texts = processor.batch_decode(
#     generated_ids,
#     skip_special_tokens=True,
# )

# print(generated_texts[0])

In [ ]:
# --- pre-extract frame manually
import cv2
import torch
from PIL import Image

def load_video_frames(path, max_frames=16):
    cap = cv2.VideoCapture(path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    indices = [int(i * total / max_frames) for i in range(max_frames)]
    frames = []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if ret:
            frames.append(Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)))
    cap.release()
    return frames

frames = load_video_frames(VIDEO_PATH)

In [ ]:
messages = [{
    "role": "user",
    "content": 
        [{"type": "image", "image": frame} for frame in frames] + [{"type": "text", "text": "You are a ultimate frisbee sports analyst. Describe this video in detail"}]
}]

In [ ]:
# ---- taking forever...
# inputs = processor.apply_chat_template(
#     messages,
#     add_generation_prompt=True,
#     tokenize=True,
#     return_dict=True,
#     return_tensors="pt",
# ).to(model.device, dtype=torch.bfloat16)

# generated_ids = model.generate(**inputs, do_sample=False, max_new_tokens=64)
# generated_texts = processor.batch_decode(
#     generated_ids,
#     skip_special_tokens=True,
# )

# print(generated_texts[0])

### Running on single frame

In [ ]:
import time
from functools import wraps

def timeit(func):
    @wraps(func)
    def timeit_wrapper(*args, **kwargs):
        start_time = time.perf_counter()
        result = func(*args, **kwargs)
        end_time = time.perf_counter()
        total_time = end_time - start_time
        print(f"Function {func.__name__} took {total_time:.4f}s")
        return result
    return timeit_wrapper


In [ ]:
@timeit
def make_inference(model, messages):
    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device, dtype=torch.bfloat16)
    
    generated_ids = model.generate(**inputs, do_sample=False, max_new_tokens=64)
    generated_texts = processor.batch_decode(
        generated_ids,
        skip_special_tokens=True,
    )
    return generated_texts

In [ ]:
messages = [{
    "role": "user",
    "content": [
        {
            "type": "image", "image": frames[0],
            "type": "text", "text": "How many players are there on the field in this picture?"
        }
    ]
}]

result = make_inference(smol_model, messages)

In [ ]:
result

In [ ]:
frames[0]

### OCR on Jersey Number

In [ ]:
from rfdetr import RFDETRSmall

rfdetr_model = RFDETRSmall()
rfdetr_model.optimize_for_inference()

cap = cv2.VideoCapture(VIDEO_PATH)

In [ ]:
ret, frame = cap.read()
img = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
# img = Image.fromarray(frame)
detections = rfdetr_model.predict(img, threshold=0.4)

In [ ]:
PERSON_ID = 1

crops = []
for coords, label in zip(detections.xyxy, detections.class_id):
    if label != PERSON_ID:
        continue
    x1, y1, x2, y2 = coords.astype(int)
    crop = frame[y1:y2, x1:x2]
    pil_img = Image.fromarray(cv2.cvtColor(crop, cv2.COLOR_RGB2BGR))
    crops.append(pil_img)

In [ ]:
# detections.xyxy.astype(int)

In [ ]:
# --- note: the number prediction is always the same
# Jersey Number => "The jersey number of the player in the picture is 10.\n\nHere's the picture:\n\nThe player is wearing a jersey with the number 10. The jersey is white with a blue stripe on the side. The player is standing in front of a white wall, and there"
# Jersey Color => "The jersey color of the player in the image is not visible in the image. The image only shows a person wearing a jersey, but the jersey color is not discernible."
# Player or referee? => always no

player_img = crops[6] # 2, 5

messages = [{
    "role": "user",
    "content": [
        {
            "type": "image", "image": player_img, 
            # "type": "text", "text": "What is the jersey number of the player in the picture?"
            # "type": "text", "text": "What is the jersey color of the player in the image?"
            "type": "text", "text": "Is the person in the picture a referee? Yes or no?"
        }
    ]
}]

result = make_inference(smol_model, messages)

In [ ]:
player_img

In [ ]:
result

### Player Teams

In [ ]:
from rfdetr import RFDETRSegSmall
from rfdetr.assets.coco_classes import COCO_CLASSES
import supervision as sv

rfdetr_seg_model = RFDETRSegSmall()

detections = rfdetr_seg_model.predict(crops[0])

In [ ]:


labels = [f"{COCO_CLASSES[class_id]}" for class_id in detections.class_id]

annotated_image = sv.MaskAnnotator().annotate(detections.data["source_image"], detections)
annotated_image = sv.LabelAnnotator().annotate(annotated_image, detections, labels)

In [ ]:
Image.fromarray(annotated_image)

In [ ]:
Image.fromarray(np.asarray(annotated_image)[y_min:y_max, :, :])


In [ ]:
# detections

In [ ]:
cropped_image = np.asarray(annotated_image)[y_min:y_max, :, :]

cropped_masks = detections.mask[:, y_min:y_max, :]

cropped_detections = detections

# shift boxes upward because image was cropped
cropped_detections.xyxy[:, [1, 3]] -= y_min

# replace masks with cropped masks
cropped_detections.mask = cropped_masks

mask_annotator = sv.MaskAnnotator()
box_annotator = sv.BoxAnnotator()

result = mask_annotator.annotate(
    scene=cropped_image.copy(),
    detections=cropped_detections
)

result = box_annotator.annotate(
    scene=result,
    detections=cropped_detections
)

sv.plot_image(result)

In [ ]:

# cropped_mask => (1, 70, 58)
# cropped_image => (70, 58, 3)
# note: broadcasting => (70, 58, 3) 

detections = rfdetr_seg_model.predict(crops[1])
h, w = detections.mask.shape[1:]

y_min = h // 4
y_max = 3 * h // 4

cropped_image = np.asarray(annotated_image)[y_min:y_max, :, :]
# cropped_mask = detections.mask[:, y_min:y_max, :]
cropped_mask = detections.mask.squeeze(0).astype(bool)
# Image.fromarray(cropped_image * cropped_mask.squeeze(0)[..., None])

# cropped_mask.squeeze(0)


In [ ]:
# detections
cropped_mask

In [ ]:
def get_player_cropped_predictions(rfdetr_seg_model, img: Image):
    detections = rfdetr_seg_model.predict(img)

    ## TODO: filter out class id that are not PERSON
    mask = detections.mask[detections.class_id == PERSON_ID]
    # print(mask.shape)
    
    s, h, w = mask.shape
    if s > 1:
        raise ValueError('more than two player detected in image')
    
    y_min = h // 4
    y_max = 3 * h // 4

    cropped_image = detections.data['source_image'][y_min:y_max, :, :]
    cropped_mask = mask[:, y_min:y_max, :]
    print(cropped_image.shape, cropped_mask.shape)

    result = cropped_image * cropped_mask.squeeze(0)[..., None]
    return result

In [ ]:
# get_cropped_predictions(rfdetr_seg_model, crops[1])

In [ ]:
# detections.data['source_image'][y_min:y_max, :, :]
# detections

In [ ]:
# 1. get cropped player torso
cropped_players = []
for pic in crops[:5]:
    cropped_players.append(get_player_cropped_predictions(rfdetr_seg_model, pic))


In [ ]:
for array in cropped_players:
    img = Image.fromarray(array)
    img.show()

In [ ]:
# ---- Finding average color vector
player_median_vectors = []
for array in cropped_players:
    med = np.median(array, axis=0)
    player_median_vectors.append(med[med.shape[0] // 2])
player_median_vectors = np.array(player_median_vectors)

fig = plt.figure()
ax = fig.add_subplot(projection='3d')
x, y, z = player_median_vectors[:, 0], player_median_vectors[:, 1], player_median_vectors[:, 2]
ax.scatter(x, y, z, c='r', marker='o')
fig.show()

In [ ]:
# using k-means to draw boundaries
from sklearn.cluster import KMeans

kmeans = KMeans(init="k-means++", n_clusters=3, n_init=4)
kmeans_labels = kmeans.fit_predict(player_median_vectors)

# cmap = plt.cm.tab10
# colors = cmap(kmeans_labels)

fig = plt.figure()
ax = fig.add_subplot(projection='3d')
x, y, z = player_median_vectors[:, 0], player_median_vectors[:, 1], player_median_vectors[:, 2]
ax.scatter(x, y, z, c=kmeans_labels, cmap='tab10', marker='o')
fig.show()

In [ ]:
# --- vectorizing using umap
import umap

# 2. resize all images to same shape
target_size = (64, 64)  # width, height

vectors = []
for img in cropped_players:
    resized = cv2.resize(img, target_size)
    vector = resized.flatten()
    vectors.append(vector)

X = np.stack(vectors)
embedding = umap.UMAP(
    n_components=2,
    n_neighbors=15, 
    min_dist=0.1, 
    metric="cosine").fit_transform(X)

In [ ]:
# embedding

In [ ]:
plt.scatter(embedding[:, 0], embedding[:, 1]) 

In [ ]:
# np.array(pic)

In [ ]:
# --- cluster only using torso